In [1]:
import pandas as pd
import sqlite3

# Tạo database trong memory
conn = sqlite3.connect(':memory:')

# Load CSV vào SQLite
ads = pd.read_csv('../data/raw/global_ads_performance_dataset.csv')
cats = pd.read_csv('../data/raw/cookie_cats.csv')

# Đưa vào SQLite như 2 bảng
ads.to_sql('ads_performance', conn, index=False, if_exists='replace')
cats.to_sql('cookie_cats', conn, index=False, if_exists='replace')

print("Database created")
print("Tables:", pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].tolist())

Database created
Tables: ['ads_performance', 'cookie_cats']


In [2]:
query1 = """
SELECT
    platform,
    COUNT(*) AS total_campaigns,
    ROUND(SUM(ad_spend), 2) AS total_spend,
    SUM(impressions) AS total_impressions,
    SUM(clicks) AS total_clicks,
    SUM(conversions) AS total_conversions,
    ROUND(AVG(ROAS), 2) AS avg_ROAS,
    ROUND(AVG(CTR), 4) AS avg_CTR,
    ROUND(AVG(CPC), 2) AS avg_CPC,
    ROUND(AVG(CPA), 2) AS avg_CPA
FROM ads_performance
GROUP BY platform
ORDER BY avg_ROAS DESC
"""
df1 = pd.read_sql(query1, conn)
print("=== PLATFORM PERFORMANCE ===")
df1

=== PLATFORM PERFORMANCE ===


,platform,total_campaigns,total_spend,total_impressions,total_clicks,total_conversions,avg_ROAS,avg_CTR,avg_CPC,avg_CPA
0,TikTok Ads,450,2653418.51,47049590,2599931,122452,9.54,0.0550,1.01,29.20
1,Meta Ads,630,2106061.67,64403494,1596736,73262,6.92,0.0250,1.32,39.10
2,Google Ads,720,6349268.91,73801150,2936149,131098,4.11,0.0399,2.15,64.06


In [3]:
query2 = """
SELECT
    platform,
    campaign_type,
    ROUND(SUM(ad_spend), 2) AS total_spend,
    ROUND(AVG(ROAS), 2) AS avg_ROAS,
    ROUND(AVG(CPA), 2) AS avg_CPA,
    SUM(conversions) AS total_conversions
FROM ads_performance
GROUP BY platform, campaign_type
ORDER BY platform, avg_ROAS DESC
"""
df2 = pd.read_sql(query2, conn)
print("=== BEST CAMPAIGN TYPES ===")
df2

=== BEST CAMPAIGN TYPES ===


,platform,campaign_type,total_spend,avg_ROAS,avg_CPA,total_conversions
0,Google Ads,Search,1629525.61,4.79,57.73,36827
1,Google Ads,Video,1664048.70,4.14,65.78,34551
2,Google Ads,Shopping,1574542.03,3.75,68.15,30419
3,Google Ads,Display,1481152.57,3.68,65.28,29301
4,Meta Ads,Display,547625.69,7.27,35.86,18634
5,Meta Ads,Search,520979.29,6.90,41.45,18339
6,Meta Ads,Shopping,501876.55,6.77,38.07,19127
7,Meta Ads,Video,535580.14,6.76,40.65,17162
8,TikTok Ads,Search,717501.95,10.52,27.31,35475
9,TikTok Ads,Display,615956.86,10.04,26.71,28934


In [4]:
query3 = """
SELECT
    version,
    COUNT(*) AS total_users,
    SUM(retention_1) AS d1_retained,
    SUM(retention_7) AS d7_retained,
    ROUND(SUM(retention_1) * 100.0 / COUNT(*), 2) AS d1_rate_pct,
    ROUND(SUM(retention_7) * 100.0 / COUNT(*), 2) AS d7_rate_pct,
    ROUND(AVG(sum_gamerounds), 2) AS avg_gamerounds
FROM cookie_cats
GROUP BY version
ORDER BY d7_rate_pct DESC
"""
df3 = pd.read_sql(query3, conn)
print("=== RETENTION BY VERSION ===")
df3

=== RETENTION BY VERSION ===


,version,total_users,d1_retained,d7_retained,d1_rate_pct,d7_rate_pct,avg_gamerounds
0,gate_30,44700,20034,8502,44.82,19.02,52.46
1,gate_40,45489,20119,8279,44.23,18.20,51.30


In [5]:
query4 = """
SELECT
    platform,
    ROUND(SUM(ad_spend), 2) AS total_spend,
    ROUND(SUM(revenue), 2) AS total_revenue,
    ROUND(SUM(revenue) / SUM(ad_spend), 2) AS overall_ROAS,
    ROUND(SUM(revenue) - SUM(ad_spend), 2) AS net_profit
FROM ads_performance
GROUP BY platform
ORDER BY overall_ROAS DESC
"""
df4 = pd.read_sql(query4, conn)
print("=== BUDGET EFFICIENCY ===")
df4

=== BUDGET EFFICIENCY ===


,platform,total_spend,total_revenue,overall_ROAS,net_profit
0,TikTok Ads,2653418.51,20223540.07,7.62,17570121.56
1,Meta Ads,2106061.67,11926045.79,5.66,9819984.12
2,Google Ads,6349268.91,22033744.95,3.47,15684476.04
